In [0]:
Very Very Important SPARK SQL Transformation
DSL approach vs declartive ( sql syntax )
performance wise no diff - dsl / sql / dsl+sql - internally its going run in RDD way
dsl + sql
read customer data
input custid,fname,lname,age,prof
custid , fullname, age , is_vote , upper(profession),datadt,createdBy
custid - direct 2.fullname - fname +lname
age = direct 4 . is_vote - derive from age column
profession - upper (profession)
data_dt- current date
createdBy- hard code -izuser

In [0]:
df=spark.sql("select current_date,'user' as user")
df.show()

In [0]:
%fs head /Volumes/izwd37dev/wd37db/rawdatta/custs_header/custs_header

In [0]:

cust_df=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/custs_header/custs_header",header=True,inferSchema=True)

cust_df.show(2)

cust_df.createOrReplaceTempView("v_cust")

spark.sql("select * from v_cust").show(2)

In [0]:
sql_str="select custid,concat(fname,' ',lname) as fullname,age,case when age >=18 then 'yes' else 'no' end as is_vote,upper(profession) as profession,current_date() as dt,'iz_user' as createdBy from v_cust"

spark.sql(sql_str).show(5)

In [0]:
cust_df.createOrReplaceTempView("v_cust")

spark.sql("select * from v_cust").show()

In [0]:
sql_str="select custid,concat(fname,' ',lname) as fullname,age,case when age >=18 then 'yes' else 'no' end as is_vote,upper(profession) as profession,current_date() as dt,'iz_user' as createdBy from v_cust"

spark.sql(sql_str).show(5)

In [0]:
spark.sql("select * from   v_cust").show(2)
spark.sql("select profession,count(*) from v_cust group by profession").show()

In [0]:
sql_str="select custid,concat(fname,' ',lname) as fullname,age,case when age >=18 then 'yes' else 'no' end as is_vote,upper(profession) as profession,current_date() as dt,'iz_user' as createdBy from v_cust"

spark.sql(sql_str).show(5)

In [0]:
spark.sql("select age,profession,count(1) as tot_count from v_cust group by age,profession order by age desc").show()

# Same using DSL Below
'''
To get both the minimum age and the count of rows per (age, profession) group, you need to include both aggregations inside .agg():
from pyspark.sql.functions import col, min, count

grouped_df = (
    cust_df.groupBy("age", "profession")
           .agg(
               min(col("age")).alias("min_age"),
               count("*").alias("row_count")
           )
)

grouped_df.show()

'''

from pyspark.sql.functions import col, min,count

grouped_df = cust_df.groupBy("age","profession").count().show()
grouped_df = cust_df.groupBy("age","profession").agg(min("age").alias("minage"),
                                                     count("*").alias("row_count")
                                                     ) 
grouped_df.show()

In [0]:
spark.sql("show tables in izwd37dev.wd37db").show()

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()

In [0]:
%sql
create view using pure sql (DDL)
load csv and create a tempview using sql syntax

In [0]:
%sql
create or replace temporary view cust_view
using csv
options (header = "true",
  inferSchema = "true",
  path = "/Volumes/izwd37dev/wd37db/rawdatta/custs_header")

In [0]:
spark.sql("select * from cust_view").show()

In [0]:
#spark.sql() = sql()
spark.sql("select * from cust_view limit 5").show()
sql("select * from cust_view limit 5").show()

In [0]:
print(sql)

In [0]:
%sql
create or replace temporary view cust_view_schema 
(
   custid int, 
   fname string,
   lname string,
   age int,
   prof string
)
using csv
options (
  header = "true",
  path = "/Volumes/izwd37dev/wd37db/rawdatta/custs_header"
);

select * from cust_view_schema limit 10

In [0]:
display(sql("show tables"))

In orrder to transform the data using sql query ,

we need to create dataframe from the source (csv/parquet/json..) using DSL is the prefred (spark.read)

df.createOrReplaceTemView using this we can create temp view

write our transform sql query and call using spark.sql or sql()

In [0]:
%sql
-- spark.read.json("/Volumes/izwd37dev/wd37db/rawdatta/emp2.json",multiline=True).createOrReplaceTempView("emp_view")

create or replace temporary view emp_view
using json
options (
    multiline = "true",
    path = "/Volumes/izwd37dev/wd37db/rawdatta/emp2.json"
);


select * from emp_view 

In [0]:
%sql
SELECT dob,to_date(dob, 'dd/MM/yyyy'),date_format(to_date(dob, 'dd/MM/yyyy'),'MM/dd/yyyy') AS parsed_date
FROM emp_view;

In [0]:
df=sql("select * from emp_view")
df.show()
df=sql("SELECT dob,CASE WHEN dob IS NULL THEN NULL ELSE to_date(dob,'dd/MM/yyyy') END AS dob_clean FROM emp_view")
df.printSchema()

df.show()

In [0]:
# reading a permannent table using sql syntax

cust_df=sql("select * from  izwd37dev.wd37db.cust_detail ")

cust_df.show()

spark.read.table("izwd37dev.wd37db.cust_detail").show()   # dsl 

In [0]:
df=sql("select * from emp_view")

df.printSchema()

df.show()


In [0]:
%sql


describe izwd37dev.wd37db.cust_detail;

describe formatted izwd37dev.wd37db.cust_detail;

describe extended izwd37dev.wd37db.cust_detail;

describe detail izwd37dev.wd37db.cust_detail;

In [0]:
cust_df.describe().show()
cust_df.summary().show()

In [0]:
spark.sql("select count(custid),min(custid),max(custid) from izwd37dev.wd37db.cust_detail ").show()
sql("select * from izwd37dev.wd37db.cust_detail").show()

## combine Data

### union , union all , unionByName(specific to DSL)


In [0]:
# union - combine same type of data / same strucure from two diff table , remove duplicate 
# union (dsl) -combine same type of data / same strucure from two diff table , all data  (similar to union all) 


# union all - combine same type of data / same strucure from two diff table , all data 

# unionByname - only available in dsl 

# in dsl , when weneed to achive the same behaviour sql union -> df.union(df2).distinct()

stud_df1=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part4.csv",header=True,inferSchema=True)
stud_df2=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part3.csv",header=True,inferSchema=True)

stud_df1.createOrReplaceTempView("stud1")
stud_df2.createOrReplaceTempView("stud2")

sql("select * from stud1").show()
sql("select * from stud2").show()

# union - SQL ( cobining two tables with same structure , positional , unique record )

sql_str="""
select sid,sname,city from stud1
union 
select sid,sname,city from stud2
"""
union_df=sql(sql_str)

union_df.show()
union_df.count()


# union All - SQL ( cobining two tables with same structure , positional , all record )

sql_str="""
select sid,sname,city from stud1
union all
select sid,sname,city from stud2
"""
union_df=sql(sql_str)

union_df.show()
union_df.count()



# unionByName - not a a option in sql 

stud_df3=spark.read.csv("/Volumes/izwd37dev/wd37db/rawdatta/student_part/stud_part2.csv",header=True,inferSchema=True)

stud_df3.createOrReplaceTempView("stud3")
# similar to union by name 
sql_str="""
select sid,sname,city,null as year from stud1
union
select sid,sname,null as city,year from stud3
"""
union_df=sql(sql_str)

union_df.show()



In [0]:
# 2. validation , cleansing , scrubbing  

### handle missing value , handling null 

 
### na.drop() -> it will null rows 

In [0]:
data=[
    (100,"crish",25),
    (101,"bala",None),
    (102,None,None),
    (None,None,None),
    (103,"raja",None),
    (104,None,25),
]


df=spark.createDataFrame(data,["id","name","age"])

df.show()

df.createOrReplaceTempView("tbl_stud")

In [0]:
df.show()
df.na.drop().show()
df.na.drop(how="any").show()

# sql 
# if all column has value retrun that record 
qry="""
select * from tbl_stud where id is not null and name is not null and age is not null
"""
sql(qry).show()

In [0]:
df.show()
df.na.drop(how="all").show()

# sql 
# if all column is null remove / ignore that record
qry="""
select * from tbl_stud where id is  not null or name is not null or age is not null
"""
sql(qry).show()

### scrubbing - filling 


nvl 

coalesce

nvl2(null,if null , if not null)

In [0]:
df.show()
df.na.fill(0).show()
df.na.fill({"age":0}).show()

df.na.fill({"age":0,"name":"NA"}).show()

# sql 

qry="""
select id,nvl(name,'NA') as name,nvl(age,0) as age from tbl_stud 
"""
sql(qry).show()

In [0]:
sql("select nvl2(null,'present','not_present')").show()

sql("select *,nvl2(age,'age provoided','get the age from user') as age_status from tbl_stud").show()



In [0]:
qry="""
select id,coalesce(name,'NA') as name,coalesce(age,0) as age from tbl_stud 
"""
sql(qry).show()

In [0]:
%sql
create or replace temporary view cust_modified_view
(
   custid int, 
   fname string,
   lname string,
   age int,
   prof string,
   erro_rec string
)
using csv
options (
  header = "true",
  path = "/Volumes/izwd37dev/wd37db/rawdatta/BB2/custsmodified",
  mode="dropmalformed",
  columnNameOfCorruptRecord="erro_rec"
);

select * from cust_modified_view limit 10

In [0]:
# add a column in DSL
# withColumn / select 

view_name="cust_modified_view"

spark.sql(f"select *,concat(fname,'~',lname) as fullname  from {view_name}").show(5)

spark.sql(f"select custid,concat(fname,'~',lname) as fullname,age,prof as job_title  from {view_name}").show(5)

spark.sql(f"select * from {view_name} limit 3").show()

In [0]:
cust_enriched_df=spark.sql(f"select custid,concat(fname,'~',lname) as fullname,age,prof as job_title,current_date() as dt,'iz_user' as user  from {view_name}")


cust_enriched_df.show(5)

In [0]:
view_name="cust_modified_view"
sql(f"select * from {view_name}").show()
spark.sql(f"select custid,upper(fname) as fname,lower(lname) as lname,initcap(concat(fname,' ',lname)) as fullname from {view_name}").show()

In [0]:
qry="select custid,fname,substring(fname,1,3) from cust_modified_view"

spark.sql(qry).show()

De duplication

In [0]:

viw_name="cust_modified_view"
# check duplicate based on custid 
spark.sql(f"select custid, count(*) from {viw_name} group by custid having count(*) > 1").show()


spark.sql("select * from cust_modified_view where custid in (4000003,4000001)").show()


spark.sql("select * from cust_modified_view where custid is null").show()

spark.sql("select * from cust_modified_view where custid in (4000003,4000001) or custid is null").show()

In [0]:
# remove duplicate 
# record level using distinct 
spark.sql(f"select count(1) from {viw_name}").show() # 10004

spark.sql(f"select distinct * from {viw_name}").count() # 9998

spark.sql(f"select count(1) from {viw_name}").show() # 10004
spark.sql(f"select * from {viw_name}").count() # 10004


spark.sql(f"select distinct * from {viw_name}").filter("custid=4000001").show()

In [0]:
# record level - DSL : distinct / dropDuplicate
# col level : DSL dropDuplicate(col)
# col level with prioriry - row_number / dropDuplicate(col)+orderBy
# if custid repeats 

df=spark.sql("select * from cust_modified_view")


# records level duplicate 

df2=df.distinct() # unique records 
df.filter("custid=4000001").show()
df2.filter("custid=4000001").show()

# column level duplicate 

dedep_col_df=df.dropDuplicates(["custid"])

dedep_col_df.filter("custid in (4000003,4000001)").show()

# sql based column level deduplication 
# if custid repeat take the latest custid (based on age desc)


In [0]:
%sql
-- sql based column level deduplication 
-- if custid repeat take the latest custid (based on age desc)

select *,row_number() over(partition by custid order by age desc) as rno from cust_modified_view;


-- inline query 
-- sub query 
select * from  (
select *,row_number() over(partition by custid order by age desc) as rno from cust_modified_view
) as tbl
where rno=1;

select * from  (
select *,row_number() over(partition by custid order by age) as rno from cust_modified_view
) as tbl
where rno=1;

In [0]:
%sql
-- cust_modified_view
-- age_cat -> if age > 60 -> senior_citizen
-- if age > 40 -> middle_age
-- others > young_age 

-- case when condition then value else value end 

select *, case when age > 60 then 'senior_citizen'
when age > 40 then 'middle_age'
else 'young_age'
end as age_cat from cust_modified_view;;


-- nvl logic using case

select * ,case when age is null then 0 else age end as n_age from cust_modified_view;



select *, case when age > 60 then 'senior_citizen'
when age >55 and prof='Lawyer' then 'spl category'
when age > 40 then 'middle_age'
else 'young_age'
end as age_cat from cust_modified_view
where age > 55 and age <60;

In [0]:
spark.sql("""
          select *, case when age > 60 then 'senior_citizen'
when age >55 and prof='Lawyer' then 'spl category'
when age > 40 then 'middle_age'
else 'young_age'
end as age_cat from cust_modified_view
where age > 55 and age <60
          """).show()

In [0]:
# udf - user defined function

def convert_upper(value):
    if value is None:
        return None
    else:
        return value.upper()
    


# test the function

print(convert_upper(None))
print(convert_upper(""))

In [0]:
cust_df=spark.sql("select * from cust_modified_view").drop("erro_rec")

cust_df.show()

# sql - udf

# step 1 - create python function 

# ste2 - register the function with spark.udf.register
spark.udf.register("sql_udf_upper",convert_upper)

cust_df=spark.sql("""select custid,fname,lname,age,prof,sql_udf_upper(nvl(fname,'')) as upper_fname ,
                  sql_udf_upper(lname) as u_lname from cust_modified_view""")

cust_df.show()

In [0]:

# lambda a:a*2
def d_age(age):
    return age*2


#udf -age -> age *2

spark.udf.register("double_age",lambda a:a*2,"int")

df=spark.sql("""select custid,fname,lname,age,prof,double_age(nvl(age,0)) as age2  from cust_modified_view""")

df.show(10)

df.printSchema()

In [0]:
# DSL : withColumn , withColumnRenamed , drop ,select , selectExpr 
# sql : select 
spark.sql("select custid,concat(fname,' ',lname) as full_name,age,current_date() as dt,'iz_user' as user from cust_modified_view").show()

# typecasting 

df=spark.sql("select custid, fname,age,cast(age as string) as age_str from cust_modified_view")
df.show(5)
df.printSchema()

## Join 

In [0]:
emp_data=[(100,"a1",25),(101,"a2",16),(102,"a3",35),(103,"a4",45),(104,"a5",25),(105,"a6",22)]

city_data=[(100,"chennai"),(105,"hyd"),(113,"delhi"),(125,"chennai")]

emp_df=spark.createDataFrame(emp_data,['id','name','age'])
city_df=spark.createDataFrame(city_data,['id','city'])

emp_df.createOrReplaceTempView("tbl_emp")
city_df.createOrReplaceTempView("tbl_city")


In [0]:
spark.sql("select * from tbl_emp").show()
spark.sql("select * from tbl_city").show()

# need all emp with matching city , emp who dont have city place it with NA

# syntax :
'''
select columns  from tbl_left join|left|right|semni|anti|full|cross tbl_right on join_condition

:left join 

select * from tbl_left left join tbl_right on join_condition
'''

qry="""
select * from tbl_emp e left join tbl_city c on e.id=c.id
"""


qry="""
select e.*,nvl(c.city,'NA') as city from tbl_emp e left join tbl_city c on e.id=c.id
"""

qry="""
select e.id,e.name,e.age,coalesce(c.city,'NA') as city from tbl_emp e left join tbl_city c on e.id=c.id
"""

spark.sql(qry).show()

In [0]:
# right join

'''
:right join 

select * from tbl_left right join tbl_right on join_condition

'''

# need  all data from city table with matching empname and age  for unmatched emp name -> unknown , age -0




In [0]:
qry="""
select c.id,nvl(e.name,'unknown') as name,nvl(e.age,0) as age,c.city from tbl_emp e right join tbl_city c on e.id=c.id
"""
spark.sql("select * from tbl_emp").show()
spark.sql("select * from tbl_city").show()
spark.sql(qry).show()

In [0]:
# full join 

qry="""
select * from tbl_emp e  full join tbl_city c on e.id=c.id
"""
spark.sql("select * from tbl_emp").show(5)
spark.sql("select * from tbl_city").show(5)
spark.sql(qry).show()

In [0]:
# semi - exists 
# anti - not exists 
# both will return only left table columns


# left semi 

# need only emp detail who provoided city in the city table

 

qry="""
select * from tbl_emp e  left semi join tbl_city c on e.id=c.id
"""
spark.sql("select * from tbl_emp").show()
spark.sql("select * from tbl_city").show()
spark.sql(qry).show()

In [0]:
# corelated subquery 

qry="""
select * from tbl_emp e where exists(select 1 from tbl_city c where e.id=c.id)
"""

spark.sql(qry).show()

# using in clause 

qry="""
select * from tbl_emp e where e.id in (select c.id from tbl_city c)
"""

spark.sql(qry).show()

In [0]:
# left anti  

# need only emp detail who is not provided the city details 

 

qry="""
select * from tbl_emp e  left anti join tbl_city c on e.id=c.id
"""
spark.sql("select * from tbl_emp").show(5)
spark.sql("select * from tbl_city").show(5)
spark.sql(qry).show()

In [0]:
# corelated subquery 
# anti join ===> not exists 
qry="""
select * from tbl_emp e where not exists(select 1 from tbl_city c where e.id=c.id)
"""

spark.sql(qry).show()

# using in clause 

qry="""
select * from tbl_emp e where e.id not in (select c.id from tbl_city c)
"""

spark.sql(qry).show()

In [0]:
# self join 

data = [
    (101,"John",104,"IT"),
    (102,"Alice",104,"IT"),
    (103,"Bob",105,"HR"),
    (104,"David",106,"IT"),
    (105,"Emma",106,"HR"),
    (106,"James",None,"Management")
]

cols=["eid","ename","mid","dept"]

emp_df=spark.createDataFrame(data,cols)

emp_df.createOrReplaceTempView("tbl_emp")

spark.sql("select * from tbl_emp").show()



# get the detail emp details 
# eid , ename , mname ,dept 

# suppose u have two table emp , mgr 

# select eid,ename,city , m.ename as mgr from emp as emp join emp as mgr on emp.mid=mgr.eid


# self join -> joining the same table 

In [0]:
%sql
select * from tbl_emp;


In [0]:
%sql
select e.eid,e.ename,nvl(m.ename,'level-1') as manager,e.dept from tbl_emp e left join tbl_emp m on e.mid=m.eid

In [0]:
%sql


select nullif(2,3)

## Aggregation / Windowing

In [0]:
cust_selected_df=spark.sql("select * from cust_modified_view")

cust_selected_df.createOrReplaceTempView("tbl_cust")

sql("select * from tbl_cust").show()

In [0]:
cust_selected_df=spark.sql("select * from cust_modified_view").drop("error_rec")

In [0]:
cust_selected_df.createOrReplaceTempView("tbl_cust")

sql("select * from tbl_cust").show()

In [0]:
qry="""
select prof,count(1) as tot_rec from tbl_cust group by prof  order by tot_rec desc
"""

display(spark.sql(qry))


# take the prof list which has more than 220 record 
# prof,tot_rec
# ordering the res based on tot_rec desc

qry="""
select prof,count(1) as tot_rec from tbl_cust group by prof  having count(1)>220 order by tot_rec desc
"""

display(spark.sql(qry))


# take prof : Politician,Photographer,Lawyer,Pilot,carpenter
# prof,tot_rec,minage , max  age 
# which ever prof has more than 100
# ordering the result by tot_rec desc


qry="""
select prof,count(1) as tot_rec,min(age) as min_age,max(age) as max_age
      from tbl_cust 
      where prof in ('Politician','Photographer','Lawyer','Pilot','carpenter')
      group by prof  having count(1)>100 
      order by tot_rec desc limit 2
"""

display(spark.sql(qry))

In [0]:
data=[(100,"abc"),(101,None),(None,None)]

cols=["id","name"]

df=spark.createDataFrame(data,cols)

df.createOrReplaceTempView("tbl_df")


spark.sql("select * from tbl_df").show()


spark.sql("select count(1),count(id),count(name) from tbl_df").show()

In [0]:
# prof rank based on the age desc order 


sql_str="""

select *,
row_number() over(partition by prof order by age desc) as row_num , 
dense_rank() over(partition by prof order by age desc) as dense_rank,
rank() over(partition by prof order by age desc) as rank from tbl_cust
where prof in ('Politician','Photographer','Lawyer','Pilot','carpenter')
"""

display(spark.sql(sql_str))


# window + agg

# window + analytical (value function)

In [0]:
# rollup , cube

# roleup
# (prof,age)--> prof +age count
#           -- prof + count
#           -- count


# df.rollup("prof","age").agg(count(1))..


qry="""

select prof,age,count(1) from tbl_cust
where prof in ('Politician','Photographer','Lawyer','Pilot','carpenter')
and age>70 
 group by rollup(prof,age) 
"""


display(spark.sql(qry))